In [17]:
import shutil  ###Libraries
import os
import math
import re
import gzip
import pandas as pd
import csv
import numpy as np

##### This function is to create wordlists of a text with no repetition

In [19]:
def extract_words(dataFrame_col):
    # Define the regex pattern with a capture group to capture words
    pattern = r'(\b\w+\b)'
    
    # Use extractall() to find all words based on the pattern
    extracted_words = dataFrame_col.str.extractall(pattern)
    
    # Flatten the result to get a list of all words
    words_list = extracted_words[0].tolist()  # Extract the first group of matches
    
    return list(set(words_list))

#### Counts the words from a column in a data frame with a certain pattern

In [21]:
def extract_words_count1(dataFrame_col):
    # Define the regex pattern with a capture group to capture words
    pattern = r'(\b\w+\b)'
    
    # Use extractall() to find all words based on the pattern
    extracted_words = dataFrame_col.str.extractall(pattern)
    
    # Flatten the result to get a list of all words
    words_list = extracted_words[0].tolist()  # Extract the first group of matches
    
    return len(words_list)

In [22]:
def extract_words_count(dataFrame_col):
    # Define the regex pattern to capture words
    pattern = r'(\b\w+\b)'
    
    # Use extractall() to find all words based on the pattern
    extracted_words = dataFrame_col.str.extractall(pattern)
    
    # Flatten the result to get a list of all words
    words_list = extracted_words[0].tolist()  # Extract the first group of matches
    
    return len(words_list)

Does as it says, given a column and a list of words it counts how many times each word appears in the column.

In [23]:
def count_words_in_column(dataFrame_col, word_list):
    # Create a dictionary to store counts for each word
    word_counts = {word.lower(): 0 for word in word_list}  # Store keys in lowercase
    
    # Create a single regex pattern for all words
    pattern = r'\b(?:' + '|'.join(re.escape(word) for word in word_list) + r')\b'
    
    # Iterate through each message in the DataFrame column
    for message in dataFrame_col:
        # Find all matches for the regex pattern in the message 
        matches = re.findall(pattern, message, flags=re.IGNORECASE)
        # Count occurrences for each matched word
        for word in matches:
            word_counts[word.lower()] += 1  # Increment the count for each matched word
    
    return word_counts

This function is for finding a file with the file name and the wd to start the search

In [14]:
def find_file(filename, search_path):
    for root, dirs, files in os.walk(search_path):
        if filename in files:
            return os.path.join(root, filename)
    return None

Here we search for the file path

In [15]:
name_of_file = 'SMSSpamCollection1'
path = find_file(name_of_file, os.path.expanduser('~'))
while path == None:
    print('File not found')
else:
    print('File Found') 

File Found


Once having the path we are aware that maybe the file will be not save as a .txt, so we create a copy of the file and save it as a text file in our working directory.

In [16]:
# Original file with non-standard extension (e.g., '.dat', '.log', etc.)
original_file = path

# New file with '.txt' extension
txt_file = "SMSSpamCollection2.txt"

# Copy the file and change the extension
shutil.copyfile(original_file, txt_file)

'SMSSpamCollection2.txt'

In [18]:
# Preprocess the file to remove any problematic quotes
with open(txt_file, 'r') as infile, open('cleaned_file.txt', 'w') as outfile:
    for line in infile:
        cleaned_line = line.replace('"', '')  # Remove quotes
        outfile.write(cleaned_line)

# Now read the cleaned file
df = pd.read_csv('cleaned_file.txt', sep='\t', header=None, names=['Category', 'Message'], engine='python')

In [24]:
df['Message'] = df['Message'].str.lower() ##Converts all the messages to lowercase letters

##df['Category'].value_counts() to see the number of ham and spam messages

##### The code partitiones the data frame in two, one corresponding to 80% of the data, which we are using to 'train' our model, and another that has the remaining 20% to see how well our model performs. All the messages(rows) are labeled\categorized as ham or spam.

In [25]:
df_80 = df.sample(frac=0.8, random_state=42)  # Randomly samples the 80% of the data
df_20 = df.drop(df_80.index) #Gives the remaining dataframe

In [28]:
all_words_list = extract_words(df_80['Message']) ##Makes the word lists from the column of messages
spam_words = extract_words_count1(df_80[df_80['Category']=='spam']['Message']) #counts numver of spam words
ham_words = extract_words_count1(df_80[df_80['Category']=='ham']['Message']) #counts number of words in ham

In [29]:
total_words = extract_words_count1(df_80['Message']) #counts total of words

In [30]:
word_list_ham = extract_words(df_80[df_80['Category'] == 'ham']['Message']) #makes list of words in ham

In [31]:
word_list_spam = extract_words(df_80[df_80['Category'] == 'spam']['Message'])#makes list of words in spam

In [32]:
word_counts_ham = count_words_in_column(df_80[df_80['Category'] == 'ham']['Message'], word_list_ham)
#counts the appearance of each word in ham

In [33]:
for word in all_words_list:
    if word not in word_counts_ham:  # Check if the word is not in the dictionary
        word_counts_ham[word] = 0  # Initialize it with a count of 0

In [34]:
word_counts_spam = count_words_in_column(df_80[df_80['Category'] == 'spam']['Message'], word_list_spam)
#counts the number of times each word appears in spam

In [35]:
for word in all_words_list:
    if word not in word_counts_spam:  # Check if the word is not in the dictionary
        word_counts_spam[word] = 0  # Initialize it with a count of 0

It is unnecesary to run the following code if you trust the process so far! However, it is never bad to check if there is anny problem. The sum of all the probabilities must add up to 1 for each ham and sum.

In [63]:
####Let's take a moment and see if everything is working out nicelly!
ham_words = extract_words_count1(df_80[df_80['Category']=='ham']['Message']) #counts number of words in ham
spam_words = extract_words_count1(df_80[df_80['Category']=='spam']['Message']) # No. of words in spam
p_word_givenham = { } ##makes a dictionary were each word is a key and the value is the probability that said word given that they are in ham
for key in word_counts_ham:
    p_word_givenham[key] =  word_counts_ham[key]/ham_words

p_word_givenspam = { } #Values are the probabilities for each word given that they are in spam
for key in word_counts_spam:
    p_word_givenspam[key] =  word_counts_spam[key]/spam_words
ham_sum = sum( p_word_givenham.values()) #adds values up
spam_sum = sum( p_word_givenspam.values()) #adds values up
print(spam_sum) ##is 1 or else....
print(ham_sum) ##is 1 or else...
#If everything is going in order, the code must prints two ones

1.0
1.0


#### What do we have so far?
Up untill this point we have the probability of spam and the probability of each word given that they are on a spam or ham message. What follows is computing the probability that a message is spam given that certain word is part of that message. We will be using the Bayes formula for this P(S|W) = [P(W|S)P(S)]/[P(W|S)P(S) + P(W|H)P(H)], also, we will be saving each probability in a dictionary keys being the word.

In [36]:
p_spam_givenword = { } ##Probability that a message is spam given that a word is in it
for word in all_words_list:
    p_spam_givenword[word] = ((word_counts_spam[word]+1)*spam_words)/(word_counts_spam[word]*spam_words + word_counts_ham[word]*ham_words)

In [37]:
p_ham_givenword = { }##Probability that a message is ham given that a word is in it
for word in all_words_list:
     p_ham_givenword[word] = ((word_counts_ham[word]+1)*ham_words)/(word_counts_spam[word]*spam_words + word_counts_ham[word]*ham_words)

In [38]:
vsp_Message = {} ##Now we calculate the v constants to calculate the probability that the message is spam
for message in df_20['Message']:
    x = extract_words_M(message)
    
    log_sum = 0
    for word in x:
        if word in p_spam_givenword:
            prob = p_spam_givenword[word]
            
            # Ensure that the probability is between 0 and 1 (exclusive) to avoid math errors
            if 0 < prob < 1:
                log_sum += abs(math.log(1 - prob)- math.log(prob))
    
    vsp_Message[message] = log_sum

In [39]:
vhm_Message = {}##Now we calculate the v constants to calculate the probability that the message is ham
for message in df_20['Message']:
    x = extract_words_M(message)
    
    log_sum = 0
    for word in x:
        if word in p_ham_givenword:
            prob = p_ham_givenword[word]
            
            # Ensure that the probability is between 0 and 1 (exclusive) to avoid math errors
            if 0 < prob < 1:
                log_sum += abs(math.log(1 - prob)- math.log(prob))
    
    vhm_Message[message] = log_sum

In [40]:
ps_Message = {} #Calculate the probability that a message in the col is spam given its words
for message, value in vsp_Message.items():
    ps_Message[message] = 1 / (1 + math.exp(value))

In [41]:
ph_Message = {} #Calculate the probability that a message in the col is ham given its words
for message, value in vhm_Message.items():
    ph_Message[message] = 1 / (1 + math.exp(value)) # Calculate based on the value

Once calculated that, we can proced to test our method with the remaining 20% of the messages

In [42]:
results = { } ##A more appropiate name would be the test messages
for message in df_20['Message']:
    if ps_Message[message] >  ph_Message[message]: #Criteria for deciding if a message is spam or not
        results[message] = {'Sim' :'spam','Real': df_20[df_20['Message'] == message]['Category'].tolist()}
    else: 
        results[message] = {'Sim':'ham', 'Real': df_20[df_20['Message'] == message]['Category'].tolist()}

In [43]:
df_results = pd.DataFrame(results) #turns results into a dataframe

In [44]:
df_results = df_results.transpose() #Makes sure df is in a col format

In [66]:
lenres = len(df_results) #prints number of rows
df_results ##Don't proceed if this does not have the correct format (in 2 columns 1087)

,Sim,Real
u dun say so early hor... u c already then say...,ham,h
"nah i don't think he goes to usf, he lives around here though",ham,h
"freemsg hey there darling it's been 3 week's now and no word back! i'd like some fun you up for it still? tb ok! xxx std chgs to send, £1.50 to rcv",ham,s
had your mobile 11 months or more? u r entitled to update to the latest colour mobiles with camera for free! call the mobile update co free on 08002986030,ham,s
oh k...i'm watching here:),ham,h
...,...,...
ic. there are a lotta childporn cars then.,ham,h
wen did you get so spiritual and deep. that's great,ham,h
have a safe trip to nigeria. wish you happiness and very soon company to share moments with,ham,h
anything lor. juz both of us lor.,ham,h


In [64]:
df_results['Real'] = df_results['Real'].str[0]

In [50]:
x = df_results[df_results['Sim'] == df_results['Real']].value_counts()
x ##Simulations what were accurate

Sim   Real
ham   ham     901
spam  spam    113
Name: count, dtype: int64

In [52]:
y = df_results[df_results['Sim'] != df_results['Real']].value_counts()
y ##Simulations that were not

Sim   Real
ham   spam    38
spam  ham     35
Name: count, dtype: int64

In [60]:
HH = x.iloc[0]
print(HH)
SS = x.iloc[1]
print(SS)
SH = y.iloc[0]
print(SH)
HS = y.iloc[1]
print(HS)

901
113
38
35


In [70]:
# Create confusion table as a DataFrame
confusion_table = pd.DataFrame({
    'Predicted Spam': [SS, HS],
    'Predicted Ham': [SH, HH]
}, index=['Actual Spam', 'Actual Ham'])
confusion_table = confusion_table*100/lenres
# Display confusion table
print(confusion_table)

             Predicted Spam  Predicted Ham
Actual Spam       10.395584       3.495860
Actual Ham         3.219871      82.888684


In [71]:
confusion_table.sum().sum()

100.0

In [73]:
error = (SH+HS)*100/lenres
error ### Calculating the error

6.7157313707451705

### Naive Bayes using sklearn

We import what we need from the sklean library

In [77]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer

In [78]:
X = df.drop(columns='Category')
y = df['Category']


# Split into training and test set (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [79]:
vectorizer = CountVectorizer()

# Fit and transform the training data (convert text to a bag of words model)
X_train_transformed = vectorizer.fit_transform(X_train['Message'])

# Transform the test data
X_test_transformed = vectorizer.transform(X_test['Message'])

# Now fit the Naive Bayes classifier
mnb = MultinomialNB()
mnb.fit(X_train_transformed, y_train)

# Predict on test data
y_pred = mnb.predict(X_test_transformed)

In [80]:
y_pred = mnb.predict(X_test_transformed)

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9856502242152466


In [83]:
# Confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
conf_matrix = conf_matrix*100/1115
print("Confusion Matrix:\n", conf_matrix)

Confusion Matrix:
 [[85.11210762  0.44843049]
 [ 0.98654709 13.4529148 ]]


In [84]:
conf_matrix.sum()

100.0

In [85]:
0.4484 + 0.9865 ##error sum of Ham classified as Spam and viseversa

1.4349